In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

BRONZE_SCHEMA = "workspace.bronze"
SILVER_SCHEMA = "workspace.silver"

expected_counts = {
    "customers": 99441,
    "geolocation": 1000163,
    "order_items": 112650,
    "order_payments": 103886,
    "order_reviews": 99224,
    "orders": 99441,
    "products": 32951,
    "sellers": 3095,
    "product_category_name_translation": 71,
}

# Convert these columns to numeric types
numeric_columns = {
    "geolocation": {
        "geolocation_lat": "double",
        "geolocation_lng": "double",
    },
    "order_items": {
        "order_item_id": "int",
        "price": "double",
        "freight_value": "double",
    },
    "order_payments": {
        "payment_sequential": "int",
        "payment_installments": "int",
        "payment_value": "double",
    },
    "order_reviews": {
        "review_score": "int",
    },
    "products": {
        "product_name_lenght": "int",
        "product_description_lenght": "int",
        "product_photos_qty": "int",
        "product_weight_g": "int",
        "product_length_cm": "int",
        "product_height_cm": "int",
        "product_width_cm": "int",
    },
}

# Convert these columns to timestamps
timestamp_columns = {
    "order_items": ["shipping_limit_date"],
    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp",
    ],
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
}

# --------------------------------------------------
# 2. Create the Silver schema
# --------------------------------------------------

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

# --------------------------------------------------
# 3. Transform each Bronze table
# --------------------------------------------------

summary_rows = []

for table_name, expected_rows in expected_counts.items():

    bronze_table = f"{BRONZE_SCHEMA}.{table_name}"
    silver_table = f"{SILVER_SCHEMA}.{table_name}"

    print(f"\nProcessing {bronze_table}")

    # Read Bronze without changing the source table
    df = spark.table(bronze_table)

    source_rows = df.count()

    # Check Bronze row counts before transformation
    if source_rows != expected_rows:
        raise ValueError(
            f"{bronze_table}: expected {expected_rows:,} rows, "
            f"found {source_rows:,}"
        )

    # Standardize strings: trim whitespace and turn
    # empty strings into NULL. Preserve metadata columns.
    for field in df.schema.fields:
        column_name = field.name

        if (
            isinstance(field.dataType, StringType)
            and not column_name.startswith("_")
        ):
            df = df.withColumn(
                column_name,
                F.when(
                    F.trim(F.col(column_name)) == "",
                    F.lit(None).cast("string"),
                ).otherwise(F.trim(F.col(column_name))),
            )

    # Convert numeric columns
    for column_name, target_type in numeric_columns.get(
        table_name, {}
    ).items():
        df = df.withColumn(
            column_name,
            F.col(column_name).cast(target_type),
        )

    # Convert timestamps. Existing missing dates remain NULL.
    for column_name in timestamp_columns.get(table_name, []):
        df = df.withColumn(
            column_name,
            F.to_timestamp(
                F.col(column_name),
                "yyyy-MM-dd HH:mm:ss",
            ),
        )

    # Remove exact duplicate geolocation records only.
    # Multiple different coordinates for the same ZIP prefix
    # are retained.
    rows_removed = 0

    if table_name == "geolocation":
        data_columns = [
            column_name
            for column_name in df.columns
            if not column_name.startswith("_")
        ]

        before_deduplication = df.count()
        df = df.dropDuplicates(data_columns)

        rows_removed = before_deduplication - df.count()

    # Write to Silver as a managed Delta table
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
    )

    silver_rows = spark.table(silver_table).count()

    summary_rows.append(
        (
            table_name,
            source_rows,
            silver_rows,
            rows_removed,
        )
    )

    print(
        f"SUCCESS: {silver_table} | "
        f"Source rows: {source_rows:,} | "
        f"Silver rows: {silver_rows:,} | "
        f"Exact duplicates removed: {rows_removed:,}"
    )

# --------------------------------------------------
# 4. Save a transformation summary
# --------------------------------------------------

summary_df = spark.createDataFrame(
    summary_rows,
    [
        "table_name",
        "bronze_rows",
        "silver_rows",
        "exact_duplicates_removed",
    ],
)

(
    summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.data_quality_summary")
)

print("\nSILVER TRANSFORMATION COMPLETED")

display(summary_df.orderBy("table_name"))



Processing workspace.bronze.customers
SUCCESS: workspace.silver.customers | Source rows: 99,441 | Silver rows: 99,441 | Exact duplicates removed: 0

Processing workspace.bronze.geolocation
SUCCESS: workspace.silver.geolocation | Source rows: 1,000,163 | Silver rows: 738,332 | Exact duplicates removed: 261,831

Processing workspace.bronze.order_items
SUCCESS: workspace.silver.order_items | Source rows: 112,650 | Silver rows: 112,650 | Exact duplicates removed: 0

Processing workspace.bronze.order_payments
SUCCESS: workspace.silver.order_payments | Source rows: 103,886 | Silver rows: 103,886 | Exact duplicates removed: 0

Processing workspace.bronze.order_reviews
SUCCESS: workspace.silver.order_reviews | Source rows: 99,224 | Silver rows: 99,224 | Exact duplicates removed: 0

Processing workspace.bronze.orders
SUCCESS: workspace.silver.orders | Source rows: 99,441 | Silver rows: 99,441 | Exact duplicates removed: 0

Processing workspace.bronze.products
SUCCESS: workspace.silver.products

table_name,bronze_rows,silver_rows,exact_duplicates_removed
customers,99441,99441,0
geolocation,1000163,738332,261831
order_items,112650,112650,0
order_payments,103886,103886,0
order_reviews,99224,99224,0
orders,99441,99441,0
product_category_name_translation,71,71,0
products,32951,32951,0
sellers,3095,3095,0


In [0]:
%sql
SHOW TABLES IN workspace.silver;

database,tableName,isTemporary
silver,customers,false
silver,data_quality_summary,false
silver,geolocation,false
silver,order_items,false
silver,order_payments,false
silver,order_reviews,false
silver,orders,false
silver,product_category_name_translation,false
silver,products,false
silver,sellers,false
